# 04 案例四：Fashion-MNIST 服饰图片生成

这个案例和 MNIST 使用几乎相同的代码，但数据分布更复杂：衣服、鞋、包的轮廓比数字更丰富。

学习重点：同一套 DDPM 代码可以迁移到另一个数据集。

In [ ]:
import math
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

torch.manual_seed(42)
random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
from torchvision import datasets, transforms

class_names = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x * 2 - 1),
])
dataset = datasets.FashionMNIST(root="../data", train=True, download=True, transform=transform)
loader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=0, drop_last=True)

In [ ]:
class DiffusionSchedule:
    def __init__(self, timesteps=300, beta_start=1e-4, beta_end=0.02, device=device):
        self.timesteps = timesteps
        self.betas = torch.linspace(beta_start, beta_end, timesteps, device=device)
        self.alphas = 1.0 - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alpha_bars = torch.sqrt(self.alpha_bars)
        self.sqrt_one_minus_alpha_bars = torch.sqrt(1.0 - self.alpha_bars)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / self.alphas)
        self.posterior_variance = self.betas * (1.0 - torch.cat([torch.ones(1, device=device), self.alpha_bars[:-1]])) / (1.0 - self.alpha_bars)


def extract(values, t, x_shape):
    batch_size = t.shape[0]
    out = values.gather(0, t)
    return out.reshape(batch_size, *((1,) * (len(x_shape) - 1)))


def q_sample(x0, t, schedule, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    return (
        extract(schedule.sqrt_alpha_bars, t, x0.shape) * x0
        + extract(schedule.sqrt_one_minus_alpha_bars, t, x0.shape) * noise
    )

In [ ]:
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None].float() * embeddings[None, :]
        return torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)

In [ ]:
class Block(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, up=False):
        super().__init__()
        self.time_mlp = nn.Linear(time_dim, out_ch)
        if up:
            self.conv1 = nn.Conv2d(2 * in_ch, out_ch, 3, padding=1)
            self.transform = nn.ConvTranspose2d(out_ch, out_ch, 4, 2, 1)
        else:
            self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
            self.transform = nn.Conv2d(out_ch, out_ch, 4, 2, 1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.bnorm1 = nn.BatchNorm2d(out_ch)
        self.bnorm2 = nn.BatchNorm2d(out_ch)

    def forward(self, x, t):
        h = self.bnorm1(F.silu(self.conv1(x)))
        time_emb = F.silu(self.time_mlp(t))
        h = h + time_emb[..., None, None]
        h = self.bnorm2(F.silu(self.conv2(h)))
        return self.transform(h)


class SimpleUnet(nn.Module):
    def __init__(self, image_channels=1, base_channels=32, time_dim=128, num_classes=None):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
        )
        self.label_emb = nn.Embedding(num_classes, time_dim) if num_classes is not None else None
        self.conv0 = nn.Conv2d(image_channels, base_channels, 3, padding=1)
        self.downs = nn.ModuleList([
            Block(base_channels, base_channels * 2, time_dim),
            Block(base_channels * 2, base_channels * 4, time_dim),
        ])
        self.ups = nn.ModuleList([
            Block(base_channels * 4, base_channels * 2, time_dim, up=True),
            Block(base_channels * 2, base_channels, time_dim, up=True),
        ])
        self.output = nn.Conv2d(base_channels, image_channels, 1)

    def forward(self, x, timestep, y=None):
        t = self.time_mlp(timestep)
        if self.label_emb is not None and y is not None:
            t = t + self.label_emb(y)

        x = self.conv0(x)
        residual_inputs = []
        for down in self.downs:
            x = down(x, t)
            residual_inputs.append(x)
        for up in self.ups:
            residual_x = residual_inputs.pop()
            x = torch.cat((x, residual_x), dim=1)
            x = up(x, t)
        return self.output(x)

In [ ]:
@torch.no_grad()
def p_sample(model, x, t, t_index, schedule, y=None):
    betas_t = extract(schedule.betas, t, x.shape)
    sqrt_one_minus_alpha_bars_t = extract(schedule.sqrt_one_minus_alpha_bars, t, x.shape)
    sqrt_recip_alphas_t = extract(schedule.sqrt_recip_alphas, t, x.shape)

    predicted_noise = model(x, t, y) if y is not None else model(x, t)
    model_mean = sqrt_recip_alphas_t * (x - betas_t * predicted_noise / sqrt_one_minus_alpha_bars_t)

    if t_index == 0:
        return model_mean
    posterior_variance_t = extract(schedule.posterior_variance, t, x.shape)
    noise = torch.randn_like(x)
    return model_mean + torch.sqrt(posterior_variance_t) * noise


@torch.no_grad()
def sample(model, shape, schedule, y=None):
    model.eval()
    x = torch.randn(shape, device=device)
    for i in tqdm(reversed(range(schedule.timesteps)), total=schedule.timesteps):
        t = torch.full((shape[0],), i, device=device, dtype=torch.long)
        x = p_sample(model, x, t, i, schedule, y=y)
    return x

In [ ]:
def show_images(x, nrow=8, title=None):
    from torchvision.utils import make_grid

    x = x.detach().cpu()
    x = (x.clamp(-1, 1) + 1) / 2
    grid = make_grid(x, nrow=nrow)
    plt.figure(figsize=(nrow, max(2, len(x) // nrow)))
    plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

In [ ]:
schedule = DiffusionSchedule(timesteps=300)
model = SimpleUnet(image_channels=1, base_channels=32, num_classes=10).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

epochs = 1
for epoch in range(epochs):
    model.train()
    pbar = tqdm(loader)
    for x0, y in pbar:
        x0 = x0.to(device)
        y = y.to(device)
        t = torch.randint(0, schedule.timesteps, (x0.shape[0],), device=device)
        noise = torch.randn_like(x0)
        xt = q_sample(x0, t, schedule, noise)
        pred_noise = model(xt, t, y)
        loss = F.mse_loss(pred_noise, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        pbar.set_description(f"epoch {epoch + 1} loss={loss.item():.4f}")

In [ ]:
labels = torch.tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9] * 2, device=device)
samples = sample(model, (len(labels), 1, 28, 28), schedule, y=labels)
show_images(samples, nrow=10, title="Fashion-MNIST conditional samples")
print(dict(enumerate(class_names)))

可以比较：
- MNIST 与 Fashion-MNIST 哪个 loss 降得更慢；
- 同样训练 1 个 epoch，哪个生成结果更清楚；
- 条件标签是否真的能控制类别。